# 01 - Before-Closure Analysis: The G-Function

# DFIT Pressure Diagnostics
Developed as part of DFIT pressure-diagnostics research in the Harold Vance Department of Petroleum Engineering at Texas A&M University.

Primary reference: Barree, Miskimins & Gilbert (2015), SPE-169539-PA.

## What this notebook covers

The G-function (Nolte, 1979) is the backbone of DFIT before-closure interpretation. It maps shut-in time onto a dimensionless variable G whose derivative reveals two things at once: the leakoff mechanism and the moment of fracture closure.

We will:
1. build the Nolte g- and G-functions for both leakoff bounds,
2. generate a synthetic DFIT,
3. compute the semilog derivative G*dP/dG, and
4. read closure off the derivative plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dfit
from dfit.derivatives import semilog_derivative, first_derivative

## 1. The g-function and G-function

Nolte's g-function has two bounding forms set by the leakoff exponent alpha: alpha = 1.0 is the high-leakoff (low-efficiency) bound and alpha = 0.5 is the low-leakoff (high-efficiency) bound. Real tests fall between the two. The G-function is G = (4/pi)[g(dtD) - g(0)], which is zero at shut-in and increases monotonically.

In [ ]:
dtd = np.linspace(0, 20, 400)
G_hi = dfit.G_function(dtd, alpha=dfit.ALPHA_HIGH_LEAKOFF)
G_lo = dfit.G_function(dtd, alpha=dfit.ALPHA_LOW_LEAKOFF)

plt.figure(figsize=(7,4))
plt.plot(dtd, G_hi, label='alpha = 1.0 (high leakoff)')
plt.plot(dtd, G_lo, label='alpha = 0.5 (low leakoff)')
plt.xlabel('dimensionless shut-in time (t - tp)/tp')
plt.ylabel('G-function')
plt.title('Nolte G-function, two leakoff bounds')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. Generate a synthetic DFIT

We synthesise a normal-leakoff test with a known closure at G = 6, so we can check our interpretation against ground truth.

In [ ]:
d = dfit.generate_dfit(regime='normal', t_pump_min=5.0,
                       isip_psi=8000, closure_pressure_psi=6800,
                       closure_G=6.0, reservoir_pressure_psi=5200,
                       seed=7)
print('ground truth:')
for k, v in d.truth.items():
    print(f'  {k}: {v}')

In [ ]:
fall = np.isfinite(d.G)
G = d.G[fall]; p = d.pressure_psi[fall]
order = np.argsort(G)
G, p = G[order], p[order]

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(d.time_min, d.pressure_psi, lw=1)
ax.set_xlabel('time since injection start (min)')
ax.set_ylabel('pressure (psi)')
ax.set_title('Synthetic DFIT: injection + falloff')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. The diagnostic plot: pressure and G*dP/dG vs G

This is the plot every DFIT analyst lives on. The semilog derivative G*dP/dG (industry convention: positive, built from the pressure decline) rises on a straight line through the origin during normal leakoff. Closure is the point where it departs below that line.

In [ ]:
sgd = semilog_derivative(G, p, smooth=5)
dpdG = first_derivative(G, p, smooth=5)

fig, ax1 = plt.subplots(figsize=(8,5))
ax1.plot(G, p, 'b-', lw=1.2, label='pressure')
ax1.set_xlabel('G-function'); ax1.set_ylabel('pressure (psi)', color='b')
ax2 = ax1.twinx()
ax2.plot(G, sgd, 'r-', lw=1.2, label='G*dP/dG')
ax2.plot(G, -G*dpdG*0 + np.nan, alpha=0)  # keep colour cycle
# reference line through origin fit to the early straight section
from numpy import polyfit
early = (G>0)&(G<=0.35*6.0)
slope = np.sum(G[early]*sgd[early])/np.sum(G[early]**2)
ax2.plot(G, slope*G, 'k--', lw=1, label='reference line')
ax2.axvline(6.0, color='g', ls=':', label='true closure G=6')
ax2.set_ylabel('G*dP/dG (psi)', color='r')
ax1.set_title('Before-closure diagnostic plot')
lines = ax1.get_lines()+ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 4. Automated closure pick and full interpretation

In [ ]:
res = dfit.analyze_dfit(d.time_min, d.pressure_psi, d.rate_bpm)
print(res.summary())

### Takeaways

- The G-function turns a messy falloff into a curve whose slope and departures carry physical meaning.
- Closure is a derivative feature, not a pressure feature - you cannot reliably pick it from the pressure curve alone.
- The closure pick is intentionally conservative; in real data closure is ambiguous by a similar margin, which is why analysts cross-check with the sqrt(t) and log-log methods (next notebooks).